In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
import warnings
warnings.filterwarnings("ignore")
import pandas as pd



In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
#

cuda


In [4]:
ROOT_DIR = "E:"
DATA_DIR = os.path.join(ROOT_DIR, "data")

dataset_path =  os.path.join(DATA_DIR, "/content/handwritten_characters.csv")

In [5]:
data = pd.read_csv(dataset_path)


In [6]:
data.head()

,Unnamed: 0,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,35.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
data.drop("Unnamed: 0",axis=1,inplace=True)

In [8]:
indx = 3

pixles = data.iloc[indx].values[1:]
pixles
label =  data.iloc[indx].values[0]
label

np.float64(15.0)

In [9]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,35.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
print(pixles.shape,label)

(784,) 15.0


In [11]:
pixles = torch.tensor(pixles,dtype=torch.float32)
label = torch.tensor(label)

In [12]:
print(pixles[230:240])
print(label)

tensor([  0.,   0.,   0.,  88., 209., 204.,  77.,   2.,   0.,   0.])
tensor(15., dtype=torch.float64)


In [13]:
pixles = pixles.reshape(28,28).unsqueeze(0)
print(pixles.shape)

torch.Size([1, 28, 28])


In [14]:
pixles_transformation = transforms.Compose([
    transforms.Normalize((0.1307,),(0.3081,))
])

pixles = pixles_transformation(pixles)
print(pixles.shape)

torch.Size([1, 28, 28])


In [15]:
class DigitDataset(torch.utils.data.Dataset):
  def __init__(self,filr_path,transform=None):
    self.data = pd.read_csv(filr_path)
    self.transform = transform

  def __len__(self):

    return len(self.data)
  def __getitem__(self, index):
    pixles = self.data.iloc[index].values[2:]
    label = self.data.iloc[index].values[1]
    pixles = torch.tensor(pixles,dtype=torch.float32)
    label = torch.tensor(label, dtype=torch.long)
    pixles = pixles.reshape(28,28).unsqueeze(0)/255
    if self.transform:
      pixles = self.transform(pixles)
    return pixles,label

In [16]:
dataset = DigitDataset(dataset_path,pixles_transformation)

In [17]:
from torch.utils.data import random_split

torch.manual_seed(42)
train_size = int(0.7*len(dataset))
val_size = int(0.15*len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset,val_dataset,test_dataset = random_split(dataset,[train_size,val_size,test_size])

In [18]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)

In [19]:
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)

In [20]:
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False)

In [21]:
print(len(train_loader))
print(len(test_loader))
print(len(val_loader))

3150
675
675


In [22]:
len(data["label"].unique())

36

In [23]:
for pixles,label in train_loader:
  print(pixles.shape)
  print(label.shape)
  break

torch.Size([32, 1, 28, 28])
torch.Size([32])


In [41]:
class digitClassifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(784,1024)
    self.fc2 = nn.Linear(1024,512)
    self.fc3 = nn.Linear(512,256)
    self.fc4 = nn.Linear(256,128)
    self.fc5 = nn.Linear(128,64)
    self.fc6 = nn.Linear(64,36)

  def forward(self,x):

    x = x.view(x.size(0), -1)
    x = self.fc1(x)
    x = torch.relu(x)
    x = self.fc2(x)
    x = torch.relu(x)
    x = self.fc3(x)
    x = torch.relu(x)
    x = self.fc4(x)
    x = torch.relu(x)
    x = self.fc5(x)
    x = torch.relu(x)
    x = self.fc6(x)
    return x

In [42]:
from torch.nn.modules import loss
def evaluate_model(model,data_loader,criterion,device):
  model.eval()
  val_loss = 0
  accuricy = 0

  with torch.no_grad():
    for batch_indx ,(pixles,label) in enumerate(data_loader):
      pixles,label = pixles.to(device),label.to(device)
      output = model(pixles)

      loss = criterion(output,label)
      val_loss += loss.item()
      _,preds = torch.max(output,dim=1)
      accuricy += torch.sum(preds==label.data)
      val_loss = val_loss/len(data_loader)
      accuricy +=(preds==label.data).sum().item()
      accuricy = accuricy/len(data_loader.dataset)
      return val_loss,accuricy


In [43]:
def traqin_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs,
    checkpoint_path
):

   model.train()
   best_val_accuricy = 0
   patience = 3

   checkpoint_dir = os.path.dirname(checkpoint_path)
   if not os.path.exists(checkpoint_dir):
     os.makedirs(checkpoint_dir)

   for epoch in range(num_epochs):
    train_loss = 0
    for batch_indx ,(pixles_batch,label_batch) in enumerate(train_loader):
      pixles_batch,label_batch = pixles_batch.to(device),label_batch.to(device)
      optimizer.zero_grad()

      output = model(pixles_batch)

      loss = criterion(output,label_batch)
      loss.backward()
      optimizer.step()
      train_loss += loss.item()
    train_loss = train_loss/len(train_loader.dataset)
    vall_loss,val_accuricy = evaluate_model(model,val_loader,criterion,device)


    if val_accuricy > best_val_accuricy:
      best_val_accuricy = val_accuricy
      torch.save(model.state_dict(),checkpoint_path)
      print("Best model saved")
    else:
      patience = patience-1
    print(f"[epoch {epoch+1}/{num_epochs},loss :{train_loss:.4f}]")
    print(f"[validation_loss {vall_loss:.4f}]")
    print(f"val_accuricy {val_accuricy:.4f}")

    if patience ==0:
      print("Model perforamche is not improving:::::")
      break



In [44]:
model = digitClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [45]:
checkpoint_path = os.path.join(
    os.getcwd(),"digit_classifier_checkpoint.pth"
)

In [47]:
traqin_model(
    model = model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=100,
    checkpoint_path=checkpoint_path
)

Best model saved
[epoch 1/100,loss :0.0181]
[validation_loss 0.0005]
val_accuricy 0.0027
[epoch 2/100,loss :0.0080]
[validation_loss 0.0004]
val_accuricy 0.0025
Best model saved
[epoch 3/100,loss :0.0061]
[validation_loss 0.0001]
val_accuricy 0.0029
[epoch 4/100,loss :0.0052]
[validation_loss 0.0002]
val_accuricy 0.0028
Best model saved
[epoch 5/100,loss :0.0045]
[validation_loss 0.0001]
val_accuricy 0.0030
[epoch 6/100,loss :0.0040]
[validation_loss 0.0004]
val_accuricy 0.0029
Model perforamche is not improving:::::
